# Day H7 — 보고와 규제, 그리고 정직한 조기 선별 baseline 미니 프로젝트

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기 — 미니 프로젝트

PADS의 실제 값을 쓰지 않고, 모양만 닮은 가상 참가자 표를 만듭니다. 특징 이름(안정 시 떨림, 좌우 비대칭, 움직임 매끄러움)과 효과 크기는 모두 가상입니다. 결과는 "PD vs PD가 아닌 사람(건강 + 다른 운동장애)"입니다. 1기 자료로 개발하고, 펌웨어가 바뀐 2기 자료로 시간적 검증을 합니다.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(42)

def cohort(n, period, gain):
    """PADS와 같은 '모양'의 가상 참가자 표: 한 사람 = 한 줄, 진단군 3개. 값은 모두 가상."""
    group = rng.choice(["Healthy", "PD", "Other"], size=n, p=[0.30, 0.45, 0.25])
    age = rng.normal(np.select([group == "PD", group == "Other"], [67, 64], 61), 9)
    female = rng.integers(0, 2, n)
    pd_ = (group == "PD").astype(float); oth = (group == "Other").astype(float)
    rest_tremor = gain * (0.9 * pd_ + 0.5 * oth + rng.normal(0, 1, n))       # 안정 시 떨림 세기(가상 단위)
    asymmetry = 0.7 * pd_ + 0.1 * oth + rng.normal(0, 1, n)                  # 좌우 손목 차이
    smoothness = -0.5 * pd_ - 0.4 * oth - 0.02 * (age - 63) + rng.normal(0, 1, n)
    return pd.DataFrame({"period": period, "group": group, "age": age.round(1), "female": female,
                         "rest_tremor": rest_tremor, "asymmetry": asymmetry, "smoothness": smoothness,
                         "y": pd_.astype(int)})

dev = cohort(450, "1기", gain=1.0)
tmp = cohort(150, "2기", gain=1.4)          # 2기: 시계 펌웨어가 바뀌어 떨림 값이 1.4배로 찍힌다고 가정
X = ["age", "female", "rest_tremor", "asymmetry", "smoothness"]
print(pd.concat([dev, tmp]).groupby(["period", "group"]).size().unstack())

# 1) 표본 수 점검(H2): 후보 변수 5개, 유병률은 개발 자료에서
prev = dev.y.mean()
lnL0 = prev * np.log(prev) + (1 - prev) * np.log(1 - prev)
r2cs = 0.15 * (1 - np.exp(2 * lnL0))
n_min = 5 / ((0.9 - 1) * np.log(1 - r2cs / 0.9))
print(f"\n개발 n={len(dev)}, PD 비율 {prev:.2f} | 기준 1(축소 0.9)의 최소 n ≈ {np.ceil(n_min):.0f}")

# 2) 내부 검증(H3): 5-fold 교차검증 예측으로 판별·보정
model = make_pipeline(StandardScaler(), LogisticRegression())
p_cv = cross_val_predict(model, dev[X], dev.y, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                         method="predict_proba")[:, 1]
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def summary(name, y, p):
    y = np.asarray(y); lp = logit(p)
    boots = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))
             if y[i].min() != y[i].max()]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    slope = sm.Logit(y, sm.add_constant(lp)).fit(disp=0).params[1]
    citl = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
    print(f"{name:12s} AUC {roc_auc_score(y, p):.3f} ({lo:.3f}–{hi:.3f}) | Brier {brier_score_loss(y, p):.3f}"
          f" | 절편 {citl:+.2f} 기울기 {slope:.2f}")
summary("내부 5-fold", dev.y, p_cv)

# 3) 시간적 검증(H6): 1기로 최종 모델, 2기에 적용
model.fit(dev[X], dev.y)
p_tmp = model.predict_proba(tmp[X])[:, 1]
summary("2기(시간적)", tmp.y, p_tmp)

# 4) 진단군별 위양성(H1의 스펙트럼): 임계값 0.5에서 PD가 아닌 사람을 양성으로 판정한 비율
for g in ["Healthy", "Other"]:
    mask = (dev.group == g).values
    print(f"내부 CV, {g:7s} 중 양성 판정 비율 {(p_cv[mask] >= 0.5).mean():.2f} (n={mask.sum()})")

# 5) 순 이득(H4): 2기 자료, 임계 확률 몇 개
def net_benefit(y, p, pt):
    y = np.asarray(y); pos = p >= pt
    return (np.sum(pos & (y == 1)) - np.sum(pos & (y == 0)) * pt / (1 - pt)) / len(y)
for pt in [0.2, 0.4, 0.6]:
    allnb = tmp.y.mean() - (1 - tmp.y.mean()) * pt / (1 - pt)
    print(f"pt {pt:.1f} | 모델 {net_benefit(tmp.y, p_tmp, pt):+.3f} | 모두 양성 {allnb:+.3f} | 아무도 0")